In [ ]:
"""ck_catan_trade_sim.py

A rough, self-contained Monte Carlo simulator for a *simplified* Catan: Cities & Knights comparison:

A) "City development" strategy:
   - Players focus on pushing ONE chosen city-improvement track (Trade/Politics/Science)
     up to TARGET_LEVEL, paying only the matching commodity.
   - Bank trading is allowed at a configurable N:1 rate, for *any* card type (resources or commodities).
   - No progress cards, robber blocking, barbarians, player-to-player trades, or harbor effects.

B) "Unit/building" strategy:
   - Using the SAME turn limit measured from (A), players try to build as many
     settlements and cities as possible (with the same N:1 bank trading).
   - We also bake a *road* cost into each new settlement (since you asked to include roads).

Other simplifications matching your spec:
- Dice: only the numbered dice (2d6). If 7 is rolled, *everyone* discards half (floor).
  (No “only if > 7 cards” threshold; no robber movement.)
- Commodity production: only cities produce commodities, and only on forest/pasture/mountains.
  City on forest/pasture/mountains yields 1 resource + 1 matching commodity.
  City on fields/hills yields 2 grain / 2 brick.
- Discard behavior:
  - Dev sim: discard resources first; if you must discard commodities, discard them randomly.
  - Unit sim: discard randomly across all cards.

"Typical starting positions" approximation:
- Each player starts with 1 city site + 1 settlement site.
- Each site has 3 adjacent hexes; each hex is sampled from a standard-ish terrain distribution
  and standard number token distribution. We pick the best of K random candidates by “pip” score.
- Setup starting hand: by default, each player gets 1 resource from each of the 3 hexes
  adjacent to their starting CITY (commodities not granted during setup).

Run:
  python ck_catan_trade_sim.py --players 4 --trade-rate 4 --target-level 4 --trials 5000

Tweak:
  - Increase --max-turns if TARGET_LEVEL is high and you see timeouts.
  - Increase --typical-samples to make starting positions stronger (and more similar).

"""

from __future__ import annotations

from dataclasses import dataclass, field
from collections import Counter
from typing import Dict, List, Optional, Tuple
import argparse
import math
import random
import statistics


# -----------------------------
# Core types / constants
# -----------------------------

RESOURCES = ("lumber", "brick", "grain", "wool", "ore")
COMMODITIES = ("paper", "cloth", "coin")
ALL_CARDS = RESOURCES + COMMODITIES

TRACKS = ("trade", "politics", "science")
TRACK_TO_COMMODITY = {
    "trade": "cloth",
    "politics": "coin",
    "science": "paper",
}

TERRAINS = ("forest", "hills", "fields", "pasture", "mountains", "desert")
TERRAIN_TO_RESOURCE = {
    "forest": "lumber",
    "hills": "brick",
    "fields": "grain",
    "pasture": "wool",
    "mountains": "ore",
    "desert": None,
}
TERRAIN_TO_COMMODITY = {
    "forest": "paper",
    "pasture": "cloth",
    "mountains": "coin",
}

# Standard number token multiset (excluding desert):
NUMBER_TOKENS = [2, 3, 3, 4, 4, 5, 5, 6, 6, 8, 8, 9, 9, 10, 10, 11, 11, 12]

# Standard-ish terrain multiset (base island): 4 forest,4 pasture,4 fields,3 hills,3 mountains,1 desert
TERRAIN_TILES = [
    "forest",
    "forest",
    "forest",
    "forest",
    "pasture",
    "pasture",
    "pasture",
    "pasture",
    "fields",
    "fields",
    "fields",
    "fields",
    "hills",
    "hills",
    "hills",
    "mountains",
    "mountains",
    "mountains",
    "desert",
]

# 2d6 distribution; we use integer weights out of 36.
DICE_WEIGHTS = {2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 5, 9: 4, 10: 3, 11: 2, 12: 1}
DICE_BAG = [v for v, w in DICE_WEIGHTS.items() for _ in range(w)]  # length 36


def pip_value(n: int) -> int:
    """Common Catan pip heuristic."""
    return {2: 1, 12: 1, 3: 2, 11: 2, 4: 3, 10: 3, 5: 4, 9: 4, 6: 5, 8: 5}.get(n, 0)


# -----------------------------
# Board abstraction
# -----------------------------

Hex = Tuple[str, Optional[int]]  # (terrain, number) ; desert has number None


@dataclass
class Site:
    """A settlement or city touching 3 hexes."""

    hexes: List[Hex]
    is_city: bool

    def produce(self, roll: int) -> Counter:
        out: Counter = Counter()
        if roll == 7:
            return out
        for terrain, num in self.hexes:
            if num is None or num != roll or terrain == "desert":
                continue
            base = TERRAIN_TO_RESOURCE[terrain]
            if base is None:
                continue
            if not self.is_city:
                out[base] += 1
                continue

            # City production in Cities & Knights:
            if terrain in ("forest", "pasture", "mountains"):
                out[base] += 1
                out[TERRAIN_TO_COMMODITY[terrain]] += 1
            else:
                # hills/fields
                out[base] += 2
        return out

    def setup_resources_from_city_placement(self) -> Counter:
        """CK setup gives 1 resource per adjacent terrain for the *placed city* (no commodities)."""
        if not self.is_city:
            return Counter()
        out: Counter = Counter()
        for terrain, num in self.hexes:
            base = TERRAIN_TO_RESOURCE[terrain]
            if base is None:
                continue
            out[base] += 1
        return out


def random_hex(rng: random.Random, avoid_desert: bool) -> Hex:
    terrain = rng.choice(TERRAIN_TILES)
    if avoid_desert and terrain == "desert":
        # Resample (small and simple).
        while terrain == "desert":
            terrain = rng.choice(TERRAIN_TILES)
    if terrain == "desert":
        return (terrain, None)
    return (terrain, rng.choice(NUMBER_TOKENS))


def random_site(
    rng: random.Random,
    is_city: bool,
    typical_samples: int = 50,
    avoid_desert: bool = True,
) -> Site:
    """Pick the best of `typical_samples` random sites by pip score (higher is 'more typical/good')."""

    best: Optional[Site] = None
    best_score = -1

    for _ in range(max(1, typical_samples)):
        hexes = [random_hex(rng, avoid_desert=avoid_desert) for _ in range(3)]
        score = sum(pip_value(num) for _, num in hexes if num is not None)
        if score > best_score:
            best_score = score
            best = Site(hexes=hexes, is_city=is_city)

    assert best is not None
    return best


# -----------------------------
# Hand / trading helpers
# -----------------------------

Cost = Dict[str, int]


def can_pay(hand: Counter, cost: Cost) -> bool:
    return all(hand[c] >= n for c, n in cost.items())


def pay(hand: Counter, cost: Cost) -> None:
    for c, n in cost.items():
        hand[c] -= n
        if hand[c] <= 0:
            del hand[c]


def _pick_trade_source(
    hand: Counter,
    trade_rate: int,
    cost: Cost,
    target_card: str,
) -> Optional[str]:
    """Choose a card type to trade away, trying not to trade away cards needed for `cost`."""
    best_src = None
    best_surplus = -10**9
    best_count = -1

    for src, cnt in hand.items():
        if src == target_card:
            continue
        if cnt < trade_rate:
            continue
        required = cost.get(src, 0)
        surplus = cnt - required
        # Prefer higher surplus; tie-break on raw count.
        if surplus > best_surplus or (surplus == best_surplus and cnt > best_count):
            best_surplus = surplus
            best_count = cnt
            best_src = src

    return best_src


def ensure_can_pay_with_trades(hand: Counter, cost: Cost, trade_rate: int) -> bool:
    """Mutates `hand` by doing N:1 bank trades to try to cover `cost`."""
    if can_pay(hand, cost):
        return True

    # Iterate each needed card type; greedily fill deficits via trades.
    for card, need in cost.items():
        while hand.get(card, 0) < need:
            src = _pick_trade_source(hand, trade_rate, cost, target_card=card)
            if src is None:
                return False
            hand[src] -= trade_rate
            if hand[src] <= 0:
                del hand[src]
            hand[card] += 1

    return can_pay(hand, cost)


def discard_half(hand: Counter, mode: str, rng: random.Random) -> None:
    """Discard floor(total/2) cards.

    mode:
      - 'bias_resources': discard resources first; if commodities must be discarded, randomize those.
      - 'random': random among all cards.
    """

    total = sum(hand.values())
    k = total // 2
    if k <= 0:
        return

    def weighted_pick(card_types: Tuple[str, ...]) -> Optional[str]:
        weights = [hand.get(c, 0) for c in card_types]
        s = sum(weights)
        if s <= 0:
            return None
        r = rng.randrange(s)
        acc = 0
        for c, w in zip(card_types, weights):
            acc += w
            if r < acc:
                return c
        return None

    if mode == "bias_resources":
        # Discard from resources as long as possible.
        for _ in range(k):
            if sum(hand.get(c, 0) for c in RESOURCES) > 0:
                c = weighted_pick(RESOURCES)
            else:
                c = weighted_pick(COMMODITIES)
            if c is None:
                break
            hand[c] -= 1
            if hand[c] <= 0:
                del hand[c]
        return

    # mode == 'random'
    for _ in range(k):
        c = weighted_pick(ALL_CARDS)
        if c is None:
            break
        hand[c] -= 1
        if hand[c] <= 0:
            del hand[c]


# -----------------------------
# Player / game state
# -----------------------------


@dataclass
class PlayerState:
    sites: List[Site]
    hand: Counter = field(default_factory=Counter)
    dev_levels: Dict[str, int] = field(default_factory=lambda: {t: 0 for t in TRACKS})

    # For the unit/build sim:
    settlements_built: int = 0  # new settlements created during sim
    cities_built: int = 0       # upgrades from settlement -> city during sim
    roads_built: int = 0        # counted only for bundled settlement+road builds

    def collect(self, roll: int) -> None:
        for s in self.sites:
            self.hand.update(s.produce(roll))

    def non_city_settlement_indices(self) -> List[int]:
        return [i for i, s in enumerate(self.sites) if not s.is_city]


# -----------------------------
# Strategies
# -----------------------------

# Build costs per your prompt.
CITY_COST: Cost = {"grain": 2, "ore": 3}
SETTLEMENT_COST: Cost = {"brick": 1, "lumber": 1, "grain": 1, "wool": 1}
ROAD_COST: Cost = {"brick": 1, "lumber": 1}

# Each new settlement requires an extra road (bundled):
SETTLEMENT_PLUS_ROAD_COST: Cost = {
    "brick": SETTLEMENT_COST["brick"] + ROAD_COST["brick"],
    "lumber": SETTLEMENT_COST["lumber"] + ROAD_COST["lumber"],
    "grain": SETTLEMENT_COST["grain"],
    "wool": SETTLEMENT_COST["wool"],
}


def choose_primary_track_by_commodity_expectation(player: PlayerState) -> str:
    """Pick the dev track aligned with the player's strongest expected commodity income.

    We estimate from current *city* sites only, using pip scores on commodity terrains.
    """
    exp = {"cloth": 0, "coin": 0, "paper": 0}
    for s in player.sites:
        if not s.is_city:
            continue
        for terrain, num in s.hexes:
            if num is None:
                continue
            if terrain in ("forest", "pasture", "mountains"):
                exp[TERRAIN_TO_COMMODITY[terrain]] += pip_value(num)

    # Map commodity -> track
    commodity_to_track = {v: k for k, v in TRACK_TO_COMMODITY.items()}
    best_comm = max(exp.items(), key=lambda kv: kv[1])[0]
    return commodity_to_track[best_comm]


def dev_turn_action(player: PlayerState, trade_rate: int, primary_track: str, target_level: int) -> None:
    """Buy as many improvements as possible in primary_track until blocked."""
    commodity = TRACK_TO_COMMODITY[primary_track]

    while player.dev_levels[primary_track] < target_level:
        next_level = player.dev_levels[primary_track] + 1
        cost = {commodity: next_level}  # 1,2,3,...

        hand_copy = player.hand.copy()
        if not ensure_can_pay_with_trades(hand_copy, cost, trade_rate):
            break

        player.hand = hand_copy
        pay(player.hand, cost)
        player.dev_levels[primary_track] = next_level


def unit_turn_action(
    player: PlayerState,
    trade_rate: int,
    rng: random.Random,
    typical_samples: int,
) -> None:
    """Build as many cities as possible, then as many settlement+road bundles as possible."""

    while True:
        built_any = False

        # 1) Upgrade settlement -> city if possible.
        settlement_idxs = player.non_city_settlement_indices()
        if settlement_idxs:
            hand_copy = player.hand.copy()
            if ensure_can_pay_with_trades(hand_copy, CITY_COST, trade_rate):
                player.hand = hand_copy
                pay(player.hand, CITY_COST)
                idx = settlement_idxs[0]
                player.sites[idx].is_city = True
                player.cities_built += 1
                built_any = True

        if built_any:
            continue

        # 2) Build a new settlement + required road (bundled cost).
        hand_copy = player.hand.copy()
        if ensure_can_pay_with_trades(hand_copy, SETTLEMENT_PLUS_ROAD_COST, trade_rate):
            player.hand = hand_copy
            pay(player.hand, SETTLEMENT_PLUS_ROAD_COST)

            # Add a new settlement site.
            player.sites.append(random_site(rng, is_city=False, typical_samples=typical_samples))
            player.settlements_built += 1
            player.roads_built += 1
            built_any = True

        if not built_any:
            break


# -----------------------------
# Simulation runners
# -----------------------------


@dataclass
class TrialResult:
    stop_turn: int
    reached: bool
    units_total: int
    units_by_player: List[int]
    cities_by_player: List[int]
    settlements_by_player: List[int]
    roads_by_player: List[int]


def _make_players(
    rng: random.Random,
    num_players: int,
    typical_samples: int,
    starting_hand: str,
) -> List[PlayerState]:
    players: List[PlayerState] = []
    for _ in range(num_players):
        city_site = random_site(rng, is_city=True, typical_samples=typical_samples)
        settlement_site = random_site(rng, is_city=False, typical_samples=typical_samples)
        p = PlayerState(sites=[city_site, settlement_site])
        if starting_hand == "ck_city":
            p.hand.update(city_site.setup_resources_from_city_placement())
        elif starting_hand == "none":
            pass
        else:
            raise ValueError(f"unknown starting_hand={starting_hand}")
        players.append(p)
    return players


def simulate_development_until_target(
    rng: random.Random,
    num_players: int,
    trade_rate: int,
    target_level: int,
    max_turns: int,
    typical_samples: int,
    starting_hand: str,
    dice_seq: List[int],
) -> Tuple[int, bool, List[str]]:
    players = _make_players(rng, num_players, typical_samples, starting_hand)

    # Choose each player's primary track.
    primaries = [choose_primary_track_by_commodity_expectation(p) for p in players]

    for t in range(1, max_turns + 1):
        roll = dice_seq[t - 1]
        if roll == 7:
            for p in players:
                discard_half(p.hand, mode="bias_resources", rng=rng)
        else:
            for p in players:
                p.collect(roll)

        active = (t - 1) % num_players
        dev_turn_action(players[active], trade_rate, primaries[active], target_level)

        if any(p.dev_levels[primaries[i]] >= target_level for i, p in enumerate(players)):
            return t, True, primaries

    return max_turns, False, primaries


def simulate_units_for_turns(
    rng: random.Random,
    num_players: int,
    trade_rate: int,
    turns: int,
    typical_samples: int,
    starting_hand: str,
    dice_seq: List[int],
) -> TrialResult:
    players = _make_players(rng, num_players, typical_samples, starting_hand)

    for t in range(1, turns + 1):
        roll = dice_seq[t - 1]
        if roll == 7:
            for p in players:
                discard_half(p.hand, mode="random", rng=rng)
        else:
            for p in players:
                p.collect(roll)

        active = (t - 1) % num_players
        unit_turn_action(players[active], trade_rate, rng=rng, typical_samples=typical_samples)

    units_by_player = [p.settlements_built + p.cities_built  for p in players] #+ p.roads_built
    return TrialResult(
        stop_turn=turns,
        reached=True,
        units_total=sum(units_by_player),
        units_by_player=units_by_player,
        cities_by_player=[p.cities_built for p in players],
        settlements_by_player=[p.settlements_built for p in players],
        roads_by_player=[p.roads_built for p in players],
    )


def run_experiment(
    trials: int,
    players: int,
    trade_rate: int,
    target_level: int,
    max_turns: int,
    typical_samples: int,
    starting_hand: str,
    seed: Optional[int] = None,
) -> None:
    rng = random.Random(seed)

    stop_turns: List[int] = []
    reached_flags: List[bool] = []
    units_totals: List[int] = []

    for _ in range(trials):
        # Common random numbers: one dice sequence per trial, used by both sims.
        dice_seq = [rng.choice(DICE_BAG) for _ in range(max_turns)]

        stop_turn, reached, _primaries = simulate_development_until_target(
            rng=rng,
            num_players=players,
            trade_rate=trade_rate,
            target_level=target_level,
            max_turns=max_turns,
            typical_samples=typical_samples,
            starting_hand=starting_hand,
            dice_seq=dice_seq,
        )

        unit_res = simulate_units_for_turns(
            rng=rng,
            num_players=players,
            trade_rate=trade_rate,
            turns=stop_turn,
            typical_samples=typical_samples,
            starting_hand=starting_hand,
            dice_seq=dice_seq,
        )

        stop_turns.append(stop_turn)
        reached_flags.append(reached)
        units_totals.append(unit_res.units_total)

    reached_rate = sum(1 for x in reached_flags if x) / len(reached_flags)

    def _pct(xs: List[int], p: float) -> float:
        xs2 = sorted(xs)
        if not xs2:
            return float("nan")
        k = max(0, min(len(xs2) - 1, int(round(p * (len(xs2) - 1)))))
        return float(xs2[k])

    mean_stop = statistics.mean(stop_turns)
    med_stop = statistics.median(stop_turns)
    p25_stop = _pct(stop_turns, 0.25)
    p75_stop = _pct(stop_turns, 0.75)

    mean_rounds = mean_stop / players
    med_rounds = med_stop / players

    mean_units = statistics.mean(units_totals)
    med_units = statistics.median(units_totals)

    print("\n=== Cities & Knights rough trade sim ===")
    print(f"players={players}  trade_rate={trade_rate}:1  target_level={target_level}  trials={trials}")
    print(f"starting_hand={starting_hand}  typical_samples={typical_samples}  max_turns={max_turns}")
    if seed is not None:
        print(f"seed={seed}")

    print("\n--- Development side (time to reach target) ---")
    print(f"reached within max_turns: {reached_rate*100:.1f}%")
    print(f"stop_turns (player-turns): mean={mean_stop:.2f}  median={med_stop:.2f}  p25={p25_stop:.0f}  p75={p75_stop:.0f}")
    print(f"stop_rounds (full rounds): mean={mean_rounds:.2f}  median={med_rounds:.2f}")

    print("\n--- Unit/building side (units built by that time) ---")
    print(f"units_total: mean={mean_units:.2f}  median={med_units:.2f}")


# -----------------------------
# CLI
# -----------------------------


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser()
    p.add_argument("--players", type=int, default=3, choices=(3, 4))
    p.add_argument("--trade-rate", type=int, default=3, dest="trade_rate")
    p.add_argument("--target-level", type=int, default=3, dest="target_level")
    p.add_argument("--trials", type=int, default=2000)
    p.add_argument("--max-turns", type=int, default=300, dest="max_turns")
    p.add_argument("--typical-samples", type=int, default=50, dest="typical_samples")
    p.add_argument("--starting-hand", type=str, default="ck_city", choices=("ck_city", "none"))
    p.add_argument("--seed", type=int, default=None)

    # Important for Colab/Jupyter: ignore unknown args (e.g., -f kernel.json)
    args, _unknown = p.parse_known_args()
    return args


def main() -> None:
    a = parse_args()
    if a.trade_rate < 2:
        raise SystemExit("--trade-rate must be >= 2")
    if a.target_level < 1:
        raise SystemExit("--target-level must be >= 1")

    run_experiment(
        trials=a.trials,
        players=a.players,
        trade_rate=a.trade_rate,
        target_level=a.target_level,
        max_turns=a.max_turns,
        typical_samples=a.typical_samples,
        starting_hand=a.starting_hand,
        seed=a.seed,
    )

if __name__ == "__main__":
    main()


=== Cities & Knights rough trade sim ===
players=3  trade_rate=3:1  target_level=3  trials=2000
starting_hand=ck_city  typical_samples=50  max_turns=300

--- Development side (time to reach target) ---
reached within max_turns: 100.0%
stop_turns (player-turns): mean=13.94  median=12.00  p25=9  p75=18
stop_rounds (full rounds): mean=4.65  median=4.00

--- Unit/building side (units built by that time) ---
units_total: mean=2.14  median=2.00


1.8566666666666667